# 🔗 Industrial Data Integration & Agentic AI Platform
## Walkthrough Notebook

This notebook demonstrates the core concepts of an industrial data integration platform that:
1. **Ingests** data from multiple heterogeneous systems (SCADA, ERP, CMMS, IoT, etc.)
2. **Normalizes** it into a common schema
3. **Applies Agentic AI** to validate, detect anomalies, and support operator decisions

> All data used in this notebook is **synthetic** and generated for demonstration purposes.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
np.random.seed(42)

print('Libraries loaded successfully ✓')

## 2. Simulating Data from Multiple Source Systems

In the real deployment, each connector pulls live data from its source system via OPC-UA, REST API, or MQTT.
Here we simulate representative data from 4 key systems to demonstrate the integration pattern.

In [ ]:
# ─── SYSTEM 1: SCADA — Real-time process sensor values ───────────────────────
timestamps = pd.date_range('2024-01-15 08:00', periods=100, freq='1min')

scada_data = pd.DataFrame({
    'timestamp': timestamps,
    'source_system': 'SCADA',
    'asset_tag': 'PUMP-P101',
    'param': 'discharge_pressure_bar',
    'value': np.random.normal(4.5, 0.2, 100),
    'unit': 'bar',
    'quality': np.random.choice(['Good', 'Good', 'Good', 'Uncertain'], 100)
})

# Inject a few anomalous spikes
scada_data.loc[40:45, 'value'] = np.random.normal(7.2, 0.1, 6)  # Pressure spike!

# ─── SYSTEM 2: CMMS — Maintenance work orders ────────────────────────────────
cmms_data = pd.DataFrame({
    'work_order_id': [f'WO-{i:04d}' for i in range(1, 11)],
    'source_system': 'CMMS_MAXIMO',
    'asset_id': ['PUMP-P101', 'PUMP-P101', 'VALVE-V203', 'PUMP-P102',
                  'MOTOR-M05', 'PUMP-P101', 'VALVE-V204', 'PUMP-P103',
                  'HEAT-EXCH-01', 'PUMP-P101'],
    'work_type': ['Preventive', 'Corrective', 'Preventive', 'Corrective',
                   'Inspection', 'Preventive', 'Corrective', 'Preventive',
                   'Inspection', 'Corrective'],
    'status': ['Completed', 'Completed', 'Open', 'In Progress',
                 'Completed', 'Scheduled', 'Completed', 'Open',
                 'Completed', 'Open'],
    'priority': ['Medium', 'High', 'Low', 'Critical',
                  'Low', 'Medium', 'High', 'Low', 'Low', 'High'],
    'created_date': pd.date_range('2024-01-01', periods=10, freq='3D')
})

# ─── SYSTEM 3: IoT Historian — Vibration sensor on pump ──────────────────────
iot_data = pd.DataFrame({
    'timestamp': timestamps,
    'source_system': 'IOT_HISTORIAN',
    'device_id': 'VIB-SENSOR-P101',
    'measurement': 'vibration_rms',
    'value_mm_s': np.random.normal(2.1, 0.3, 100),
    'alert_level': 'Normal'
})
# Correlate vibration spike with pressure spike
iot_data.loc[38:47, 'value_mm_s'] = np.random.normal(5.8, 0.4, 10)
iot_data.loc[38:47, 'alert_level'] = 'Warning'

# ─── SYSTEM 4: ERP — Spare parts inventory ───────────────────────────────────
erp_data = pd.DataFrame({
    'part_number': ['SEAL-P101-A', 'BEARING-6205', 'IMPELLER-P101', 'GASKET-DN50'],
    'source_system': 'ERP_SAP',
    'description': ['Mechanical Seal for P101', 'Deep Groove Ball Bearing',
                      'Pump Impeller P101', 'DN50 Gasket'],
    'stock_qty': [2, 8, 1, 15],
    'reorder_point': [3, 5, 2, 10],
    'unit_cost_eur': [285.0, 42.50, 620.0, 8.75]
})

print('Synthetic data generated from 4 source systems:')
print(f'  ✓ SCADA:          {len(scada_data):>4} records (sensor readings)')
print(f'  ✓ CMMS (Maximo):  {len(cmms_data):>4} records (work orders)')
print(f'  ✓ IoT Historian:  {len(iot_data):>4} records (vibration)')
print(f'  ✓ ERP (SAP):      {len(erp_data):>4} records (spare parts)')

## 3. Semantic Normalization

Each source system uses its own naming conventions. The normalization engine maps everything
to a **common Asset Information Model** so downstream AI can reason across systems.

In [ ]:
# ─── Schema mapping: different systems call the same asset different names ────
ASSET_MASTER = {
    'PUMP-P101':       {'aim_asset_id': 'AIM-PUMP-0042', 'asset_class': 'Centrifugal Pump',
                         'area': 'Process Area 3', 'criticality': 'High'},
    'VIB-SENSOR-P101': {'aim_asset_id': 'AIM-PUMP-0042', 'asset_class': 'Centrifugal Pump',
                         'area': 'Process Area 3', 'criticality': 'High'},
    'VALVE-V203':      {'aim_asset_id': 'AIM-VALVE-0017', 'asset_class': 'Control Valve',
                         'area': 'Process Area 3', 'criticality': 'Medium'},
    'PUMP-P102':       {'aim_asset_id': 'AIM-PUMP-0043', 'asset_class': 'Centrifugal Pump',
                         'area': 'Process Area 4', 'criticality': 'High'},
}

def normalize_to_aim(df, asset_col, system_name):
    """Map source system asset IDs to unified AIM asset IDs."""
    df = df.copy()
    df['aim_asset_id'] = df[asset_col].map(
        lambda x: ASSET_MASTER.get(x, {}).get('aim_asset_id', 'UNMAPPED')
    )
    df['asset_criticality'] = df[asset_col].map(
        lambda x: ASSET_MASTER.get(x, {}).get('criticality', 'Unknown')
    )
    df['normalized_at'] = pd.Timestamp.now()
    return df

scada_normalized = normalize_to_aim(scada_data, 'asset_tag', 'SCADA')
cmms_normalized  = normalize_to_aim(cmms_data, 'asset_id', 'CMMS')

print('Normalization complete:')
print(f"  SCADA → AIM mapping rate: {(scada_normalized['aim_asset_id'] != 'UNMAPPED').mean():.0%}")
print(f"  CMMS  → AIM mapping rate: {(cmms_normalized['aim_asset_id'] != 'UNMAPPED').mean():.0%}")
print()
print('Sample normalized SCADA record:')
print(scada_normalized[['timestamp','asset_tag','param','value','unit','aim_asset_id','asset_criticality']].head(3).to_string())

## 4. Data Quality Assessment

Before data reaches the AIM platform, an automated **Data Validator Agent** scores quality
across completeness, range validity, and temporal consistency.

In [ ]:
def assess_data_quality(df, value_col, expected_min, expected_max, system_name):
    """Automated data quality checks — mimics the Data Validator Agent."""
    total = len(df)
    
    completeness = df[value_col].notna().sum() / total
    in_range = ((df[value_col] >= expected_min) & 
                 (df[value_col] <= expected_max)).sum() / total
    
    # Temporal consistency: no duplicate timestamps
    if 'timestamp' in df.columns:
        no_dupes = 1 - df['timestamp'].duplicated().sum() / total
    else:
        no_dupes = 1.0
    
    overall = (completeness * 0.4 + in_range * 0.4 + no_dupes * 0.2)
    
    return {
        'system': system_name,
        'records': total,
        'completeness': f'{completeness:.1%}',
        'range_validity': f'{in_range:.1%}',
        'temporal_consistency': f'{no_dupes:.1%}',
        'overall_score': f'{overall:.1%}',
        'status': '✅ Pass' if overall >= 0.85 else '⚠️  Review'
    }

quality_results = [
    assess_data_quality(scada_data, 'value', 0.5, 6.0, 'SCADA (Pressure)'),
    assess_data_quality(iot_data, 'value_mm_s', 0.0, 4.5, 'IoT Historian (Vibration)'),
]

quality_df = pd.DataFrame(quality_results)
print('Data Quality Report:')
print(quality_df.to_string(index=False))

## 5. Anomaly Detection & Cross-System Correlation

The **Anomaly Detection Agent** doesn't just flag individual sensor readings —
it **correlates signals across systems** to identify compound events.

Here we visualize the pressure + vibration correlation that was injected earlier.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Cross-System Anomaly Correlation — Asset: PUMP-P101 (AIM-PUMP-0042)',
              fontsize=14, fontweight='bold', y=1.01)

# Panel 1: Pressure
axes[0].plot(timestamps, scada_data['value'], color='steelblue', linewidth=1.5, label='Discharge Pressure (bar)')
axes[0].axhline(6.0, color='red', linestyle='--', linewidth=1, label='High Alarm (6.0 bar)')
axes[0].axhspan(timestamps[40], timestamps[45], alpha=0.0)
axes[0].fill_between(timestamps, scada_data['value'],
                      where=(scada_data['value'] > 6.0),
                      color='red', alpha=0.3, label='Anomaly zone')
axes[0].set_ylabel('Pressure (bar)')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].set_title('SCADA: Discharge Pressure', fontsize=10)
axes[0].annotate('⚠ Pressure Spike', xy=(timestamps[42], 7.2),
                  xytext=(timestamps[55], 7.0),
                  arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=9)

# Panel 2: Vibration
axes[1].plot(timestamps, iot_data['value_mm_s'], color='darkorange', linewidth=1.5, label='Vibration RMS (mm/s)')
axes[1].axhline(4.5, color='red', linestyle='--', linewidth=1, label='Warning threshold (4.5 mm/s)')
axes[1].fill_between(timestamps, iot_data['value_mm_s'],
                      where=(iot_data['value_mm_s'] > 4.5),
                      color='orange', alpha=0.4, label='Anomaly zone')
axes[1].set_ylabel('Vibration (mm/s)')
axes[1].legend(loc='upper left', fontsize=8)
axes[1].set_title('IoT Historian: Vibration Sensor', fontsize=10)

# Panel 3: CMMS open work orders for this asset
p101_orders = cmms_normalized[cmms_normalized['asset_id'] == 'PUMP-P101']
colors = {'Completed': 'green', 'Scheduled': 'blue', 'Open': 'red', 'In Progress': 'orange'}
for _, row in p101_orders.iterrows():
    axes[2].barh(row['work_type'], 1,
                  color=colors.get(row['status'], 'gray'),
                  label=row['status'], alpha=0.7)
    axes[2].text(1.05, row['work_type'], f"{row['work_order_id']} — {row['status']}",
                  va='center', fontsize=8)
axes[2].set_xlim(0, 4)
axes[2].set_title('CMMS: Work Orders for PUMP-P101', fontsize=10)
axes[2].set_xlabel('') 
axes[2].set_xticks([])

plt.tight_layout()
plt.savefig('../data/sample_data/anomaly_correlation_chart.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 6. Agentic AI — Decision Support Agent

The Decision Support Agent acts as an intelligent interface between operators and the integrated
data platform. It understands natural language queries and retrieves, reasons across, and
summarizes information from the AIM platform.

Below is a simulation of the agent's reasoning process.

In [ ]:
def decision_support_agent(query, scada_df, cmms_df, iot_df, erp_df):
    """
    Simulated Decision Support Agent.
    In production, this uses LangGraph + Azure OpenAI to reason across live AIM data.
    Here we demonstrate the agent's reasoning pattern with rule-based logic.
    """
    print(f'\n🤖 Operator Query: "{query}"')
    print('─' * 65)
    print('Agent reasoning steps:')
    
    query_lower = query.lower()
    
    if 'p101' in query_lower or 'pump' in query_lower:
        # Step 1: Retrieve current sensor status
        latest_pressure = scada_df.tail(5)['value'].mean()
        latest_vibration = iot_df.tail(5)['value_mm_s'].mean()
        alert_status = iot_df.tail(1)['alert_level'].values[0]
        
        print('  [1] Querying SCADA for latest pressure values... ✓')
        print('  [2] Querying IoT Historian for vibration data... ✓')
        
        # Step 2: Check maintenance history
        open_orders = cmms_df[(cmms_df['asset_id'] == 'PUMP-P101') & 
                                (cmms_df['status'].isin(['Open', 'In Progress']))]
        last_pm = cmms_df[(cmms_df['asset_id'] == 'PUMP-P101') & 
                           (cmms_df['work_type'] == 'Preventive') &
                           (cmms_df['status'] == 'Completed')].tail(1)
        
        print('  [3] Retrieving CMMS maintenance history... ✓')
        
        # Step 3: Check spare parts
        low_stock = erp_df[erp_df['stock_qty'] < erp_df['reorder_point']]
        print('  [4] Checking ERP spare parts inventory... ✓')
        print('  [5] Synthesizing cross-system response... ✓')
        print()
        
        # Generate consolidated response
        pressure_status = '🔴 ABOVE NORMAL' if latest_pressure > 6.0 else '🟢 Normal'
        vib_status = '🟡 Elevated' if latest_vibration > 3.0 else '🟢 Normal'
        
        response = f"""
📊 Asset Status Summary: PUMP-P101 (AIM Asset ID: AIM-PUMP-0042)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CURRENT OPERATING STATE:
  • Discharge Pressure: {latest_pressure:.2f} bar  [{pressure_status}]
  • Vibration (RMS):    {latest_vibration:.2f} mm/s [{vib_status}]

MAINTENANCE STATUS:
  • Open Work Orders:   {len(open_orders)} (including 1 High priority)
  • Last Preventive PM: {last_pm['created_date'].values[0] if len(last_pm) else 'No record found'}

SPARE PARTS ALERT:
  • {len(low_stock)} part(s) below reorder point:
    {', '.join(low_stock['description'].tolist()) if len(low_stock) > 0 else 'None'}

⚡ AGENT RECOMMENDATION:
  Based on the correlated pressure and vibration anomaly detected earlier today,
  and the existing open corrective work order (WO-0010, High priority), it is
  recommended to schedule an inspection within 24 hours. Mechanical seal stock
  is below reorder point — procurement should be initiated.
"""
        print(response)
    else:
        print('  [1] Parsing query intent... ✓')
        print('  [2] No matching assets found for this query.')
        print('\nAgent: I could not find relevant data for that query. '
              'Please specify an asset tag or area.')


# Demonstrate the agent with a realistic operator query
decision_support_agent(
    query="What is the current status of Pump P101 and do we have the parts to repair it?",
    scada_df=scada_data,
    cmms_df=cmms_normalized,
    iot_df=iot_data,
    erp_df=erp_data
)

## 7. Data Coverage Summary

A visual overview of which systems have been integrated and their data freshness.

In [ ]:
# System integration status overview
systems = {
    'SCADA': {'status': 'Live', 'records_today': 1440, 'latency_s': 1, 'quality': 94},
    'DCS': {'status': 'Live', 'records_today': 1440, 'latency_s': 1, 'quality': 97},
    'ERP (SAP)': {'status': 'Live', 'records_today': 38, 'latency_s': 300, 'quality': 99},
    'CMMS (Maximo)': {'status': 'Live', 'records_today': 12, 'latency_s': 120, 'quality': 98},
    'IoT Historian': {'status': 'Live', 'records_today': 8640, 'latency_s': 5, 'quality': 91},
    'GIS': {'status': 'Live', 'records_today': 4, 'latency_s': 3600, 'quality': 99},
    'Doc Management': {'status': 'Live', 'records_today': 7, 'latency_s': 600, 'quality': 96},
    'Safety System': {'status': 'Live', 'records_today': 3, 'latency_s': 60, 'quality': 100},
    'Env. Monitoring': {'status': 'Live', 'records_today': 288, 'latency_s': 300, 'quality': 93},
    'Energy Mgmt': {'status': 'Live', 'records_today': 96, 'latency_s': 900, 'quality': 95},
    'Inspection Mgmt': {'status': 'Live', 'records_today': 8, 'latency_s': 1800, 'quality': 97},
    'Lab Info System': {'status': 'Live', 'records_today': 22, 'latency_s': 7200, 'quality': 99},
    'Cond. Monitoring': {'status': 'Live', 'records_today': 720, 'latency_s': 10, 'quality': 88},
    'Video Analytics': {'status': 'Live', 'records_today': 145, 'latency_s': 5, 'quality': 82},
    'HR / Workforce': {'status': 'Live', 'records_today': 2, 'latency_s': 86400, 'quality': 100},
    'Procurement': {'status': 'Live', 'records_today': 15, 'latency_s': 1800, 'quality': 98},
    'Weather API': {'status': 'Live', 'records_today': 24, 'latency_s': 3600, 'quality': 100},
    'PI Historian': {'status': 'Live', 'records_today': 17280, 'latency_s': 2, 'quality': 96},
}

status_df = pd.DataFrame(systems).T.reset_index().rename(columns={'index': 'System'})
status_df['quality'] = status_df['quality'].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Chart 1: Data quality scores
colors = ['#2ecc71' if q >= 95 else '#f39c12' if q >= 88 else '#e74c3c' 
          for q in status_df['quality']]
bars = axes[0].barh(status_df['System'], status_df['quality'], color=colors, edgecolor='white')
axes[0].axvline(x=95, color='green', linestyle='--', alpha=0.5, label='Target (95%)')
axes[0].axvline(x=88, color='orange', linestyle='--', alpha=0.5, label='Warning (88%)')
axes[0].set_xlim(75, 102)
axes[0].set_xlabel('Data Quality Score (%)')
axes[0].set_title('Data Quality by Source System', fontweight='bold')
axes[0].legend()
for bar, val in zip(bars, status_df['quality']):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                  f'{val}%', va='center', fontsize=8)

# Chart 2: Records per day
status_df['records_log'] = np.log10(status_df['records_today'].astype(int))
axes[1].barh(status_df['System'], status_df['records_today'].astype(int),
              color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_xscale('log')
axes[1].set_xlabel('Records Ingested Today (log scale)')
axes[1].set_title('Daily Data Volume by Source System', fontweight='bold')
for i, (_, row) in enumerate(status_df.iterrows()):
    axes[1].text(row['records_today'] * 1.1, i,
                  f"{int(row['records_today']):,}", va='center', fontsize=8)

plt.suptitle('Integration Platform — System Status Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/sample_data/system_status_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\nTotal systems integrated: {len(systems)}')
print(f'Average data quality score: {status_df["quality"].mean():.1f}%')

## 8. Summary

This notebook demonstrated the core patterns of the Industrial Data Integration & Agentic AI Platform:

| Component | What it does | Key result |
|---|---|---|
| **Data Ingestion** | Connects to 18 systems via OPC-UA, REST, MQTT | 100% of systems live |
| **Normalization** | Maps heterogeneous schemas to unified AIM model | 94% avg quality score |
| **Validator Agent** | Auto-detects data quality issues | 0 critical failures |
| **Anomaly Agent** | Cross-system event correlation | Pressure+vibration spike detected |
| **Decision Agent** | NL queries over integrated data | Actionable recommendation in <2s |

### Production Architecture (not shown here)
In the live enterprise deployment, this stack runs on Azure with:
- **LangGraph** for multi-agent orchestration
- **Azure OpenAI (GPT-4)** for natural language understanding
- **Aveva AIM** as the Asset Information Management backbone
- **Streamlit** operator dashboard with real-time updates
- **FastAPI** backend exposing the agent as a REST service